In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [2]:
# =========================
# 1. CSV 로드
# =========================
csv_path = "/ssd1/jueon/wj/detoxicity_model/ace_safe_ver/splits/scaffold_by_endpoint_property_outlier_dropped/split_summary.csv" 
df_stats = pd.read_csv(csv_path)

# 숫자형 컬럼 보정
num_cols = ["n_total", "n_valid_smiles", "n_train", "n_valid", "n_test"]
for col in num_cols:
    df_stats[col] = pd.to_numeric(df_stats[col], errors="coerce").fillna(0).astype(int)

In [3]:
df_stats.head()

,dataset_name,endpoint,smiles_col,split_method,split_type,n_total,n_valid_smiles,n_train,n_valid,n_test,note
0,ames,ames,toxic_safe_decoded_smiles,scaffold,train_test,6444,6444,5867,0,577,"9:1 scaffold, min_test=30"
1,clintox,clintox,toxic_safe_decoded_smiles,scaffold,unseen_endpoint_test,41,41,0,0,41,n_total<=75; full test
2,dictrank,dictrank,toxic_safe_decoded_smiles,scaffold,unseen_endpoint_test,26,26,0,0,26,n_total<=75; full test
3,dilist,dilist,toxic_safe_decoded_smiles,scaffold,train_test,87,87,57,0,30,"9:1 scaffold, min_test=30"
4,diril,diril,toxic_safe_decoded_smiles,scaffold,unseen_endpoint_test,7,7,0,0,7,n_total<=75; full test


In [4]:
total_test_num = 0
for n_test_num in df_stats['n_total']:
    total_test_num += n_test_num
total_test_num

52882

In [5]:
total_test_num = 0
for n_test_num in df_stats['n_test']:
    total_test_num += n_test_num
total_test_num

4914

In [6]:
total_train_num = 0
for n_train_num in df_stats['n_train']:
    total_train_num += n_train_num
total_train_num

47968

In [7]:
# =========================
# 2. dataset별 요약 통계
# =========================
dataset_summary = (
    df_stats.groupby("dataset_name", as_index=False)
    .agg(
        n_endpoints=("endpoint", "count"),
        total_samples=("n_total", "sum"),
        total_train=("n_train", "sum"),
        total_valid=("n_valid", "sum"),
        total_test=("n_test", "sum"),
    )
    .sort_values("total_samples", ascending=False)
)

print("=== Dataset별 요약 ===")
print(dataset_summary.to_string(index=False), "\n")

=== Dataset별 요약 ===
 dataset_name  n_endpoints  total_samples  total_train  total_valid  total_test
     tox21_df           12          30240        28324            0        1916
   metabolism            5           8508         7660            0         848
         ames            1           6444         5867            0         577
 herg_unified            1           5360         4824            0         536
        sider           27           2060         1157            0         903
skin_reaction            1            109           79            0          30
       dilist            1             87           57            0          30
      clintox            1             41            0            0          41
     dictrank            1             26            0            0          26
        diril            1              7            0            0           7 



In [8]:
# =========================
# 3. endpoint별 요약 통계
# =========================
endpoint_summary = (
    df_stats[["dataset_name", "endpoint", "n_total", "n_train", "n_valid", "n_test", "split_type"]]
    .sort_values(["n_total", "dataset_name"], ascending=[False, True])
)

print("=== Endpoint별 요약 (n_total 큰 순) ===")
print(endpoint_summary.to_string(index=False),"\n")

=== Endpoint별 요약 (n_total 큰 순) ===
 dataset_name                                                            endpoint  n_total  n_train  n_valid  n_test           split_type
         ames                                                                ames     6444     5867        0     577           train_test
     tox21_df                                                         tox21_NR-ER     6430     6111        0     319           train_test
 herg_unified                                                        herg_unified     5360     4824        0     536           train_test
     tox21_df                                                        tox21_NR-AhR     4643     4408        0     235           train_test
     tox21_df                                                        tox21_SR-MMP     4499     4178        0     321           train_test
     tox21_df                                                        tox21_SR-ARE     3143     2857        0     286           train_test

In [9]:
# =========================
# 4. 작은 endpoint 개수 확인
# =========================
bins_summary = {
    "n_total < 5": (df_stats["n_total"] < 5).sum(),
    "5 <= n_total < 10": ((df_stats["n_total"] >= 5) & (df_stats["n_total"] < 10)).sum(),
    "10 <= n_total < 30": ((df_stats["n_total"] >= 10) & (df_stats["n_total"] < 30)).sum(),
    "30 <= n_total < 100": ((df_stats["n_total"] >= 30) & (df_stats["n_total"] < 100)).sum(),
    "n_total >= 100": (df_stats["n_total"] >= 100).sum(),
}
print("=== Endpoint 크기 구간별 개수 ===")
for k, v in bins_summary.items():
    print(f"{k}: {v}")
print()

=== Endpoint 크기 구간별 개수 ===
n_total < 5: 1
5 <= n_total < 10: 1
10 <= n_total < 30: 3
30 <= n_total < 100: 17
n_total >= 100: 29



In [10]:
# =========================
# 6. imbalance 확인용 비율 컬럼
# =========================
df_stats["train_ratio"] = df_stats["n_train"] / df_stats["n_total"].replace(0, 1)
df_stats["valid_ratio"] = df_stats["n_valid"] / df_stats["n_total"].replace(0, 1)
df_stats["test_ratio"]  = df_stats["n_test"]  / df_stats["n_total"].replace(0, 1)

print("=== Split ratio 예시 ===")
print(
    df_stats[
        ["dataset_name", "endpoint", "n_total", "train_ratio", "valid_ratio", "test_ratio"]
    ].sort_values("n_total", ascending=False).head(20).to_string(index=False)
)

=== Split ratio 예시 ===
dataset_name                           endpoint  n_total  train_ratio  valid_ratio  test_ratio
        ames                               ames     6444     0.910459          0.0    0.089541
    tox21_df                        tox21_NR-ER     6430     0.950389          0.0    0.049611
herg_unified                       herg_unified     5360     0.900000          0.0    0.100000
    tox21_df                       tox21_NR-AhR     4643     0.949386          0.0    0.050614
    tox21_df                       tox21_SR-MMP     4499     0.928651          0.0    0.071349
    tox21_df                       tox21_SR-ARE     3143     0.909004          0.0    0.090996
    tox21_df                    tox21_NR-ER-LBD     3119     0.931388          0.0    0.068612
  metabolism                      cyp2c19_veith     2147     0.900326          0.0    0.099674
    tox21_df                       tox21_SR-HSE     2087     0.935793          0.0    0.064207
    tox21_df               